###  Day 24 – Model Robustness & Stress Testing
- Noise injection
- Performance degradation analysis
- Robustness evaluation

In [2]:
import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

In [4]:
model = joblib.load("calibrated_rf.pkl")
threshold = joblib.load("best_cost_threshold.pkl")

X_test = joblib.load("X_test.pkl")
y_test = joblib.load("y_test.pkl")

In [5]:
y_proba_original = model.predict_proba(X_test)[:, 1]
y_pred_original = (y_proba_original >= threshold).astype(int)

acc_original = accuracy_score(y_test, y_pred_original)
f1_original = f1_score(y_test, y_pred_original)

print("Original Accuracy:", acc_original)
print("Original F1:", f1_original)

Original Accuracy: 0.6984732824427481
Original F1: 0.6108374384236454


In [6]:
np.random.seed(42)

X_test_noisy = X_test.copy()
noise = np.random.normal(0, 0.05, X_test.shape)

X_test_noisy = X_test_noisy + noise

In [7]:
y_proba_noisy = model.predict_proba(X_test_noisy)[:, 1]
y_pred_noisy = (y_proba_noisy >= threshold).astype(int)

acc_noisy = accuracy_score(y_test, y_pred_noisy)
f1_noisy = f1_score(y_test, y_pred_noisy)

print("Noisy Accuracy:", acc_noisy)
print("Noisy F1:", f1_noisy)

Noisy Accuracy: 0.683206106870229
Noisy F1: 0.5870646766169154


In [8]:
acc_drop = acc_original - acc_noisy
f1_drop = f1_original - f1_noisy

print("Accuracy Drop:", acc_drop)
print("F1 Drop:", f1_drop)

Accuracy Drop: 0.01526717557251911
F1 Drop: 0.023772761806729936


In [9]:
print("Original Confusion Matrix:\n", confusion_matrix(y_test, y_pred_original))
print("\nNoisy Confusion Matrix:\n", confusion_matrix(y_test, y_pred_noisy))

Original Confusion Matrix:
 [[121  68]
 [ 11  62]]

Noisy Confusion Matrix:
 [[120  69]
 [ 14  59]]


### The noise injection test shows only a minor degradation in performance (Accuracy drop ≈ 0.015 and F1 drop ≈ 0.024). The confusion matrix comparison indicates that a small number of true positives shifted to false negatives under noisy conditions, slightly reducing recall. However, the overall change is limited, suggesting that the calibrated Random Forest model remains reasonably robust to small perturbations in the feature space and maintains stable decision behavior.